# 02 — Stage-1 Self-Supervised Pretraining (`MA-UGM` + `SACR`)

Pretrains `Pan-Organ Net V2` with Modality-Adaptive Uncertainty-Guided Masking (EMA Teacher) and Self-Supervised Anatomical Coordinate Regression under simulated partial-FOV cropping. Automatically resumes from Google Drive checkpoints across Colab sessions.


In [ ]:
import os, sys, shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = Path('/content/drive/MyDrive/ResearchPilot_PanOrgan/V2')
CODE_DIR = DRIVE_ROOT / 'code'
SHARDS_DIR = DRIVE_ROOT / 'preprocessed_shards'
CKPT_DIR = DRIVE_ROOT / 'checkpoints'
EXP_DIR = DRIVE_ROOT / 'experiments'
for d in [SHARDS_DIR, CKPT_DIR, EXP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Unpack code package onto local fast SSD if uploaded as tarball, or install directly
if (CODE_DIR / 'V2_code_package.tar.gz').exists():
    Path('/content/panorgan_v2').mkdir(parents=True, exist_ok=True)
    !tar -xzf "{CODE_DIR}/V2_code_package.tar.gz" -C /content/panorgan_v2

PKG_ROOT = Path('/content/panorgan_v2') if Path('/content/panorgan_v2/pyproject.toml').exists() else CODE_DIR
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))
!pip install -q --no-deps -e "{PKG_ROOT}"
!pip install -q timm nibabel pydicom pyarrow scikit-learn scipy PyYAML tqdm open-clip-torch kaggle

import importlib
importlib.invalidate_caches()
import torch
from panorgan.config import load_config, detect_runtime_profile, seed_everything

cfg = load_config(PKG_ROOT / 'configs' / 'v2_panorgan.yaml')
seed_everything(cfg['project']['seed'], cfg['project']['deterministic'])
runtime = detect_runtime_profile(cfg)
print(f'Active Runtime -> GPU: {runtime.gpu_name} | Tier: {runtime.tier} | Batch: {runtime.batch_size}x{runtime.grad_accum} | AMP: {runtime.amp_dtype}')


In [ ]:
import pandas as pd
from torch.utils.data import DataLoader
from panorgan.data import BalancedModalityBatchSampler, PanOrganShardDataset, ShardCacheManager
from panorgan.models import PanOrganNetV2
from panorgan.engine import ColabResilientTrainer

manifest_df = pd.read_parquet(SHARDS_DIR / 'master_manifest.parquet')
train_df = manifest_df[manifest_df['split'] == 'train'].reset_index(drop=True)
val_df = manifest_df[manifest_df['split'] == 'val'].reset_index(drop=True)
print(f'SSL Train samples: {len(train_df)} | Val samples: {len(val_df)}')

mod_map = {k: int(v['id']) for k, v in cfg['modalities'].items()}
cache_mgr = ShardCacheManager(
    drive_shards_root=SHARDS_DIR,
    local_cache_dir=Path(cfg['storage']['local_cache_dir']),
    max_cache_gb=float(cfg['storage']['max_local_cache_gb']),
)
cache_mgr.warmup_all_shards()

train_ds = PanOrganShardDataset(train_df, cache_mgr, mod_map, target_hw=(256, 256), augment=True, mnar_truncation=0.25)
val_ds = PanOrganShardDataset(val_df, cache_mgr, mod_map, target_hw=(256, 256), augment=False, mnar_truncation=0.0)

sampler = BalancedModalityBatchSampler(
    train_df['dataset'].tolist(),
    batch_size=runtime.batch_size,
    temperature_alpha=cfg['training']['stage2_multitask']['sampling_temperature'],
)
pin_mem = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_sampler=sampler, num_workers=cfg['storage']['num_workers'], pin_memory=pin_mem)
val_loader = DataLoader(val_ds, batch_size=runtime.batch_size, shuffle=False, num_workers=cfg['storage']['num_workers'], pin_memory=pin_mem)

model = PanOrganNetV2.from_config(cfg)
ssl_cfg = cfg['training']['stage1_ssl']
trainer = ColabResilientTrainer(
    model=model,
    runtime=runtime,
    checkpoint_dir=CKPT_DIR / 'stage1_ssl_ma_ugm',
    lr=float(ssl_cfg['lr']),
    weight_decay=float(ssl_cfg['weight_decay']),
    sacr_loss_weight=float(ssl_cfg['sacr_loss_weight']),
    moe_aux_weight=float(cfg['model']['moe_aux_weight']),
)
history = trainer.fit(train_loader, val_loader, num_epochs=int(ssl_cfg['epochs']), mode='ssl', warmup_epochs=int(ssl_cfg['warmup_epochs']), resume=True)
print('Stage-1 SSL completed. Best Val Loss:', trainer.best_val_loss)
